# Local Government Spending
By function. Cities and counties report spending in functional groups that SCO redefined in FY2016-17; this page maps both eras onto one set of groups using the finest reported label (see `src/ca_taxes/local.py`). City utilities, sanitation and transportation moved between categories at the break, so treat jumps at the dotted line as reporting, not policy.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
SCO_NOTE = "Source: California State Controller's Office, Cities / Counties / Special Districts Annual Reports"
BREAK = {2016.5: "SCO reporting format changes"}
ls = pd.read_csv(PROCESSED / "local_spending.csv")
for govt in ["City", "County"]:
    d = add_views(ls[ls.govt == govt].groupby(["fy", "group"], as_index=False).amount_k.sum())
    stacked_area_toggle(d, "group", f"{govt} spending by function", SCO_NOTE, breaks=BREAK).show()

## Special districts
Water, power, hospital, fire, transit and other special districts, by activity (top 12 by cumulative spending).

In [2]:
sd = pd.read_csv(PROCESSED / "local_sd_activity.csv")
sp = sd[sd.kind == "spending"]
top = sp.groupby("activity").amount_k.sum().sort_values(ascending=False).head(12).index
sp = sp.assign(activity=sp.activity.where(sp.activity.isin(top), "All other activities"))
d = add_views(sp.groupby(["fy", "activity"], as_index=False).amount_k.sum())
stacked_area_toggle(d, "activity", "Special district spending by activity", SCO_NOTE).show()